# STIC model validation

Validation of the model using field data from the ameriflux database and landsat acquisitions

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import re
import datetime as dt
from collections import deque
from itertools import repeat
from dateutil import tz

import numpy as np
import numpy.typing as npt
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from numba import njit
from plotly.subplots import make_subplots
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sweat.common.flux import CST_SB
from sweat.common.utils import compute_vari_green_index
from sweat.stic.constant import PT_CST, PSYCHROMETRIC_CST, KELVIN_CST
from sweat.stic.convert import convert_kelvin_to_celsius, convert_to_rh, convert_to_local_time
from sweat.stic.models.functions import _tetens, _tetens_derivative
from sweat.common.flux import compute_et_from_le
from sweat.stic.runner import run_batch_model

In [ ]:
if os.environ.get("SWEAT_TEST_DATA_PATH", None) is None:
    msg = "Variable SWEAT_TEST_DATA_PATH must be set"
    raise ValueError(msg)

## Functions

In [ ]:
# Functions for handling EC data

def get_code(name:str)-> str:
    """
    Get code site from the name
    """
    m = re.match(r'US-(.*)[_,-].*', name)
    if m:
        name = m.group(1)
    if len(name) >3:
        name = name[0:3]
    return name

def get_ec_data_filename(name:str, root:str)-> str:
    """
    Get EC data path for a site
    """
    code = get_code(name)
    files = [f for f in os.listdir(root) if re.match(r'AMF_US-(.*)_BASE_HH_.*\.csv', f)]
    for file in files:
        if code.lower() in file.lower():
            return os.path.join(root,file)
    raise IOError(f"File {file} for ground truth data not found")

def read_ec_data(name:str, root: str)-> pd.DataFrame:
    """
    Read EC data file for a site
    """
    filename = get_ec_data_filename(name,root)
    # Read
    df = pd.read_csv(filename,
                    sep=",",
                    encoding="ascii", 
                    skiprows=2, 
                    usecols=lambda col: col in ["TIMESTAMP_START","TIMESTAMP_END", "G","TA"] or col.startswith("TS") or col.startswith("TA_") or col.startswith("RH") or col.startswith("G_PI_F") or col.startswith("H_PI_F") or col.startswith("LE_PI_F") or col.startswith("NETRAD") or col.startswith("SW_IN") or col.startswith("SW_OUT"),
                    parse_dates=["TIMESTAMP_START","TIMESTAMP_END"])
    #print(f"Retrieve columns: {df.columns})
    # Rename columns
    df = df.rename(columns={"TIMESTAMP_START":"start","TIMESTAMP_END":"end"})
    # LE
    best_le = min(
    (c for c in df.columns if c.startswith("LE_PI_F")),
    key=lambda c: (0,) if c == "LE_PI_F" else (1, *map(int, c.split("_")[3:]))
    )
    df = df.rename(columns={best_le: "ec_le"})
    # H
    best_h = min(
    (c for c in df.columns if c.startswith("H_PI_F")),
    key=lambda c: (0,) if c == "H_PI_F" else (1, *map(int, c.split("_")[3:]))
    )
    df = df.rename(columns={best_h: "ec_h"})
    # RN
    if df.columns.str.startswith("NETRAD_PI_F").any():
        best_rn = min(
        (c for c in df.columns if c.startswith("NETRAD_PI_F")),
        key=lambda c: (0,) if c == "NETRAD_PI_F" else (1, *map(int, c.split("_")[3:]))
        )
    else:
        best_rn = min(
        (c for c in df.columns if c.startswith("NETRAD") and not c.startswith("NETRAD_PI_F")),
        key=lambda c: (0,) if c == "NETRAD" else (1, *map(int, c.split("_")[3:]))
        )
    df = df.rename(columns={best_rn: "ec_rn"})
    # SW IN
    if df.columns.str.startswith("SW_IN_PI_F").any():
        best_sw_in = min(
        (c for c in df.columns if c.startswith("SW_IN_PI_F")),
        key=lambda c: (0,) if c == "SW_IN_PI_F" else (1, *map(int, c.split("_")[4:]))
        )
    else:
        best_sw_in = min(
        (c for c in df.columns if c.startswith("SW_IN") and not c.startswith("SW_IN_PI_F")),
        key=lambda c: (0,) if c == "SW_IN" else (1, *map(int, c.split("_")[3:]))
        )
    df = df.rename(columns={best_sw_in: "ec_sw_in"})
    # SW OUT
    if df.columns.str.startswith("SW_OUT_PI_F").any():
        best_sw_out = min(
        (c for c in df.columns if c.startswith("SW_OUT_PI_F")),
        key=lambda c: (0,) if c == "SW_OUT_PI_F" else (1, *map(int, c.split("_")[4:]))
        )
    else:
        best_sw_out = min(
        (c for c in df.columns if c.startswith("SW_OUT") and not c.startswith("SW_OUT_PI_F")),
        key=lambda c: (0,) if c == "SW_OUT" else (1, *map(int, c.split("_")[3:]))
        )
    df = df.rename(columns={best_sw_out: "ec_sw_out"})
    # G
    if df.columns.str.startswith("G_PI_F").any():
        best_g = min(
        (c for c in df.columns if c.startswith("G_PI_F")),
        key=lambda c: (0,) if c == "G_PI_F" else (1, *map(int, c.split("_")[3:]))
        )
        df = df.rename(columns={best_g: "ec_g"})
    elif "G" in df.columns:
        best_g = "G"
        df = df.rename(columns={"G": "ec_g"})
    else:
        print(f"No G flux for {name}")
        best_g = "G"
        df["ec_g"] = 0.0
    # TA
    best_ta = min(
    (c for c in df.columns if c.startswith("TA")),
    key=lambda c: (0,) if c == "TA" else (1, *map(int, c.split("_")[3:]))
    )
    df = df.rename(columns={best_ta: "ec_ta"})
    # TS
    best_ts = min(
    (c for c in df.columns if c.startswith("TS") and not c.startswith("TS_PI")),
    key=lambda c: (0,) if c == "TS" else (1, *map(int, c.split("_")[3:]))
    )
    df = df.rename(columns={best_ts: "ec_ts"})
    # RH
    best_rh = min(
    (c for c in df.columns if c.startswith("RH")),
    key=lambda c: (0,) if c == "RH" else (1, *map(int, c.split("_")[3:]))
    )
    df = df.rename(columns={best_rh: "ec_rh"})
    print(f"Site {name} used data = {best_le} {best_h} {best_g} {best_rn} {best_sw_in} {best_sw_out} {best_ta} {best_ts} {best_rh}")
    df = df[["start","end","ec_le","ec_h","ec_g","ec_rn","ec_sw_out","ec_sw_in","ec_ta","ec_ts","ec_rh"]]
    # Exclude nodata
    value_cols = df.columns.difference(["start","end"])
    # replace -9999 with NaN only in value columns
    df[value_cols] = df[value_cols].replace(-9999, np.nan)
    # drop rows where any value column is NaN
    df = df.dropna(subset=value_cols, how="any")
    # Compute EF
    df["ec_ef"] = df["ec_le"]/(df["ec_h"]+df["ec_le"])
    # Compute ET
    df["ec_et"] = df.apply(lambda x: compute_et_from_le(x["ec_le"]),axis=1)
    # Compute solar net radiation
    df["ec_srn"] = df["ec_sw_in"] - df["ec_sw_out"]
    # Compute dewpoint temperature
    eastar = 6.13753*np.exp((17.27*df["ec_ta"])/(df["ec_ta"]+237.3))   
    ea = (df["ec_rh"]/100)*(eastar)
    df["ec_td"] = 237.3*np.log(ea/6.13753)/(17.27 - np.log(ea/6.13753))
    # Compute Rn - G
    df["ec_rn-g"] = df["ec_rn"] - df["ec_g"]
    # Compute LE closed
    df["ec_le_closed"] = df["ec_rn"] - df["ec_g"] - df["ec_h"]
    # Keep columns
    df = df[["start","end","ec_le", "ec_h","ec_ef","ec_g", "ec_rn", "ec_srn", "ec_ta","ec_ts","ec_td","ec_rh", "ec_et","ec_rn-g","ec_le_closed"]]
    return df

def get_ec_data(df:pd.DataFrame, path: str) -> tuple[pd.DataFrame,pd.DataFrame]:
    """
    Get EC data at acquisition dates
    """
    # Get sites
    sites = list(df["name"].unique())
    # Loop for each site
    datasets = []
    ec_datasets = []
    for site in sites:
        print(site)
        # Select data for a site
        selected_df = df[df["name"] == site].reset_index(drop=True)
        # Read
        ec_df = read_ec_data(site,path)
        # Search indexes
        intervals = pd.IntervalIndex.from_arrays(ec_df["start"].dt.tz_localize(None), ec_df["end"].dt.tz_localize(None), closed="both")
        intervals = intervals.astype("interval[datetime64[ns]]")
        matches,not_matches = intervals.get_indexer_non_unique(selected_df["date"].dt.tz_localize(None))
        # Remove no matches
        matches = matches[matches!=-1]
        datasets.append(selected_df.drop(not_matches).reset_index(drop=True))
        ec_datasets.append(pd.concat([selected_df[["name","date"]].drop(not_matches).reset_index(drop=True), ec_df.iloc[matches].drop(columns=["start","end"]).reset_index(drop=True)],axis=1))
        print(f"Site {site} processed")
    return pd.concat(datasets), pd.concat(ec_datasets)
    

In [ ]:
# Functions for reading input data

def get_info(filename:str)-> pd.DataFrame:
    """
    Get information
    """
    return pd.read_excel(filename,sheet_name="INFO")

def read_input_data(filename:str) -> pd.DataFrame:
    """
    Read input data
    """
    sites = pd.ExcelFile(filename).sheet_names
    sites.remove("INFO")
    return pd.concat([df for _,df in pd.read_excel(filename,sheet_name=sites).items()])

def correct_data(df: pd.DataFrame, ec_df: pd.DataFrame, variables:list[str]) -> pd.DataFrame:
    """
    Use EC data instead
    """
    ec_variables = [f"ec_{v}" for v in variables]
    ec_input = ec_df[["date","name"] + ec_variables]
    corrected_df = pd.merge(left=df, right=ec_input,on=["date","name"])
    return corrected_df.drop(columns=variables).rename(columns={k:v for k,v in zip(ec_variables,variables)})


In [ ]:
# Functions for metrics

def compute_metrics(measured: npt.ArrayLike, estimated: npt.ArrayLike)-> tuple[float,float,float,float,float]:
    """
    Compute slope, mbe, mae, rmse, r2
    """
    idx = np.isfinite(measured) & np.isfinite(estimated)
    slope,_ = np.polyfit(measured[idx], estimated[idx], 1)
    mbe = np.nanmean(estimated-measured)
    mae = mean_absolute_error(measured[idx], estimated[idx])
    rmse = np.sqrt(mean_squared_error(measured[idx], estimated[idx]))
    r2 = r2_score(measured[idx], estimated[idx])
    return (slope,mbe,mae,rmse,r2)

def search_data(df:pd.DataFrame,name:str,date:str)-> pd.DataFrame:
    """ 
    Search data
    """ 
    date_dt = dt.datetime.strptime(date,"%Y-%m-%d")
    query = f"name.str.contains('{name}') and date >= '{date_dt:%Y-%m-%d} 00:00:00' and date<'{date_dt+dt.timedelta(days=1):%Y-%m-%d} 00:00:00' "
    return df.query(query)

def plot_metrics(df: pd.DataFrame, variable: str = "le"):
    """
    Interactive plot with Plotly (hover shows date)
    """

    groups = list(df.name.unique())
    groups.append("all")
    n_groups = len(groups)

    # assign one color per group (excluding "all")
    base_groups = df.name.unique()
    colors = px.colors.qualitative.Plotly

    color_map = {
    g: colors[i % len(colors)]
    for i, g in enumerate(base_groups)
    }

    ncols = 3
    nrows = int(np.ceil(n_groups / ncols))

    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=groups,
        horizontal_spacing=0.06,
        vertical_spacing=0.14
    )

    for i, group in enumerate(groups):
        row = i // ncols + 1
        col = i % ncols + 1

        if group == "all":
            measured = df[f"ec_{variable}"]
            estimated = df[variable]
            for g in df.name.unique():
                subset = df[df.name == g]

                fig.add_trace(
                    go.Scatter(
                        x=subset[f"ec_{variable}"],
                        y=subset[variable],
                        mode="markers",
                        marker=dict(
                            size=6,
                            opacity=0.5,
                            color=color_map[g]
                        ),
                        name=g,
                        legendgroup=g,
                        showlegend=(group=="all"),  # only show legend here if you want
                        customdata=subset["date"],
                        hovertemplate=(
                            "Group: " + g + "<br>"
                            "Measured: %{x:.3f}<br>"
                            "Estimated: %{y:.3f}<br>"
                            "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                        )
                    ),
                    row=row,
                    col=col
                )
        else:
            subset = df[df.name == group]
            measured = subset[f"ec_{variable}"]
            estimated = subset[variable]

            # Scatter with hover info (including date)
            fig.add_trace(
                go.Scatter(
                    x=measured,
                    y=estimated,
                    mode="markers",
                    marker=dict(size=6, opacity=0.6,color=color_map.get(group, "gray")),
                    name=group,
                    showlegend=False,
                    customdata=subset["date"],
                    hovertemplate=(
                        "Measured: %{x:.2f}<br>"
                        "Estimated: %{y:.2f}<br>"
                        "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                    )
                ),
                row=row,
                col=col
            )

        # Identity line
        lims = [
            min(measured.min(), estimated.min()),
            max(measured.max(), estimated.max())
        ]

        fig.add_trace(
            go.Scatter(
                x=lims,
                y=lims,
                mode="lines",
                line=dict(dash="dash", color="black"),
                showlegend=False
            ),
            row=row,
            col=col
        )

        # Linear fit
        idx = np.isfinite(measured) & np.isfinite(estimated)
        slope,intercept = np.polyfit(measured[idx], estimated[idx], 1)
        fig.add_trace(
            go.Scatter(
                x=lims,
                y=[lims[0] * slope + intercept, lims[1] * slope + intercept],
                mode="lines",
                line=dict(color="red"),
                showlegend=False
            ),
            row=row,
            col=col
        )

        # Metrics
        slope_m, mbe, mae, rmse, r2 = compute_metrics(measured, estimated)

        fig.add_annotation(
            x=0.97,
            y=0.03,
            xref="x domain",
            yref="y domain",
            text=(
                f"slope = {slope_m:.2f}<br>"
                f"MBE = {mbe:.2f}<br>"
                f"MAE = {mae:.2f}<br>"
                f"RMSE = {rmse:.2f}<br>"
                f"R2 = {r2:.2f}"
            ),
            showarrow=False,
            align="right",
            row=row,
            col=col,
        )

    fig.update_xaxes(title_text=f"Measured {variable.upper()}")
    fig.update_yaxes(title_text=f"Estimated {variable.upper()}")
    fig.update_layout(
        height=400 * nrows,
        width=500 * ncols,
        title=f"Measured vs Estimated {variable.upper()}",
    )

    fig.show()


In [ ]:
def plot_data(df: pd.DataFrame,variable:str):
    """
    Plot data
    """
    fig = px.line(
        df,
        x="date",
        y=variable,
        color="name",   # group by "name"
        markers=True,
        custom_data=["name"],
    )

    # Print value
    fig.update_traces(
    hovertemplate=
    "<b>Site:</b> %{customdata[0]}<br>" +
    "<b>Date:</b> %{x|%Y-%m-%d}<br>" +
    f"<b>{variable}:</b> %{{y:.2f}}<br>" +
    "<extra></extra>"
    )

    # Format x-axis as YYYY-MM-DD
    fig.update_xaxes(
        tickformat="%Y-%m-%d"
    )

    # Improve layout
    fig.update_layout(
        xaxis_title="Date",
        yaxis_title=variable,
        legend_title="Site",
        title={
        "text": f"Evolution of {variable} over time",
        "x": 0.5,
        "xanchor": "center",
        "font": {"size": 20}
       }
    )

    fig.show() 

def plot_scatter(df: pd.DataFrame, x: str, y: str, range: tuple[float,float] | None=None):
    """
    Interactive plot with Plotly (hover shows date)
    """

    groups = list(df.name.unique())
    groups.append("all")
    n_groups = len(groups)

    # assign one color per group (excluding "all")
    base_groups = df.name.unique()
    colors = px.colors.qualitative.Plotly

    color_map = {
    g: colors[i % len(colors)]
    for i, g in enumerate(base_groups)
    }

    ncols = 3
    nrows = int(np.ceil(n_groups / ncols))

    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=groups,
        horizontal_spacing=0.06,
        vertical_spacing=0.14
    )

    for i, group in enumerate(groups):
        row = i // ncols + 1
        col = i % ncols + 1


        if group == "all":
            for g in df.name.unique():
                subset = df[df.name == g]

                fig.add_trace(
                    go.Scatter(
                        x=subset[x],
                        y=subset[y],
                        mode="markers",
                        marker=dict(
                            size=6,
                            opacity=0.5,
                            color=color_map[g]
                        ),
                        name=g,
                        legendgroup=g,
                        showlegend=(group=="all"),  # only show legend here if you want
                        customdata=subset["date"],
                        hovertemplate=(
                            "Group: " + g + "<br>"
                            f"{x}: %{{x:.3f}}<br>"
                            f"{y}: %{{y:.3f}}<br>"
                            "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                        )
                    ),
                    row=row,
                    col=col
                )
        else:
            subset = df[df.name == group]

            xdata = subset[x]
            ydata = subset[y]
            # Scatter with hover info (including date)
            fig.add_trace(
                go.Scatter(
                    x=xdata,
                    y=ydata,
                    mode="markers",
                    marker=dict(size=6, opacity=0.6,color=color_map.get(group, "gray")),
                    name=group,
                    showlegend=False,
                    customdata=subset["date"],
                    hovertemplate=(
                        f"{x}: %{{x:.3f}}<br>"
                        f"{y}: %{{y:.3f}}<br>"
                        "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                    )
                ),
                row=row,
                col=col
            )

    fig.update_xaxes(title_text=f"{x}")
    fig.update_yaxes(title_text=f"{y}")
    if range is not None:
        fig.update_xaxes(range=range)
        fig.update_yaxes(range=range)
    fig.update_layout(
        height=400 * nrows,
        width=500 * ncols,
        title=f"{x} vs {y}",
    )

    fig.show()

def plot_compare(df1: pd.DataFrame, df2: pd.DataFrame, x: str, y: str, range: tuple[float,float] | None=None):
    """
    Interactive plot with Plotly (hover shows date)
    """

    groups = list(df1.name.unique())
    groups.append("all")
    n_groups = len(groups)

    # assign one color per group (excluding "all")
    base_groups = df1.name.unique()
    colors = px.colors.qualitative.Plotly

    color_map = {
    g: colors[i % len(colors)]
    for i, g in enumerate(base_groups)
    }

    ncols = 3
    nrows = int(np.ceil(n_groups / ncols))

    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=groups,
        horizontal_spacing=0.06,
        vertical_spacing=0.14
    )

    for i, group in enumerate(groups):
        row = i // ncols + 1
        col = i % ncols + 1


        if group == "all":
            for g in df1.name.unique():
                subset1 = df1[df1.name == g]
                subset2 = df2[df2.name == g]

                fig.add_trace(
                    go.Scatter(
                        x=subset1[x],
                        y=subset2[y],
                        mode="markers",
                        marker=dict(
                            size=6,
                            opacity=0.5,
                            color=color_map[g]
                        ),
                        name=g,
                        legendgroup=g,
                        showlegend=(group=="all"),  # only show legend here if you want
                        customdata=subset1["date"],
                        hovertemplate=(
                            "Group: " + g + "<br>"
                            f"{x}: %{{x:.3f}}<br>"
                            f"{y}: %{{y:.3f}}<br>"
                            "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                        )
                    ),
                    row=row,
                    col=col
                )
        else:
            subset1 = df1[df1.name == group]
            subset2 = df2[df2.name == group]

            xdata = subset1[x]
            ydata = subset2[y]
            # Scatter with hover info (including date)
            fig.add_trace(
                go.Scatter(
                    x=xdata,
                    y=ydata,
                    mode="markers",
                    marker=dict(size=6, opacity=0.6,color=color_map.get(group, "gray")),
                    name=group,
                    showlegend=False,
                    customdata=subset1["date"],
                    hovertemplate=(
                        f"{x}: %{{x:.3f}}<br>"
                        f"{y}: %{{y:.3f}}<br>"
                        "Date: %{customdata|%Y-%m-%d}<extra></extra>"
                    )
                ),
                row=row,
                col=col
            )

    fig.update_xaxes(title_text=f"{x}")
    fig.update_yaxes(title_text=f"{y}")
    if range is not None:
        fig.update_xaxes(range=range)
        fig.update_yaxes(range=range)
    fig.update_layout(
        height=400 * nrows,
        width=500 * ncols,
        title=f"{x} vs {y}",
    )

    fig.show()

## Input

In [ ]:
stic_data_path = os.path.join(os.environ.get("SWEAT_TEST_DATA_PATH"),"stic")
input_data_path = os.path.join(stic_data_path,"California_2017-2025_PointScale.xlsx")

## Get information on input data

Input content

In [ ]:
info_df = get_info(input_data_path)

In [ ]:
info_df

## Read data

In [ ]:
input_data = read_input_data(input_data_path)

In [ ]:
input_data

## Prepare data

* Convert temperatures to Celsius degrees.
* Compute local time in seconds taking into account the Landsat's overpass over the area around 18h45 GMT, i.e. UTC.
* Compute long wave net radiation with the following formula :
$$
L_{net} = \epsilon * \sigma_{SB} * 1.24 * (ea/(T_a+273.15))^{1/7}*(T_a+273.15)^4 - \epsilon*\sigma_{SB}*(LST+273.15)^4
$$
* Compute net radiation from **ssr** and **lnet**

In [ ]:
def prepare_input_data(df:pd.DataFrame) -> pd.DataFrame:
    """
    Prepare data to run STIC
    """
    # ts, ta, td
    output_df = df[["st_b10","t2m","d2m"]].apply(convert_kelvin_to_celsius).rename(columns={"st_b10":"ts","t2m":"ta","d2m":"td"})
    # rh
    output_df["rh"] = output_df[["ta","td"]].apply(lambda row: convert_to_rh(row["ta"],row["td"]),axis=1)
    # fc
    output_df["fc"] = df["fcover"]
    # lai
    output_df["lai"] = df["lai"]
    # ln
    eastar = output_df["ta"].apply(_tetens)
    ea = output_df["rh"] * eastar / 100
    output_df["ln"] = df["st_emis"] * CST_SB * 1.24 * (ea/(output_df["ta"]+KELVIN_CST))**(1/7)*(output_df["ta"]+KELVIN_CST)**4 - df["st_emis"]*CST_SB*(output_df["ts"]+KELVIN_CST)**4
    # rn
    output_df["rn"] = df["ssr"] + output_df["ln"]
    output_df["srn"] = df["ssr"]
    # localtime
    output_df["local_time"] = df[["Landsat_overpass","Lon","Lat"]].apply(lambda row: convert_to_local_time(date=dt.datetime.combine(row["Landsat_overpass"].date(),dt.time(hour=18,minute=45),tzinfo=dt.UTC),y=row["Lat"],x=row["Lon"],crs="epsg:4326"),axis=1)
    # name
    output_df["name"] = df["Name"]
    # lat/lon
    output_df["lon"] = df["Lon"]
    output_df["lat"] = df["Lat"]
    # Reflectance
    output_df["nir"] = df["sr_b5"]
    output_df["swir"] = df["sr_b6"]
    output_df["blue"] = df["sr_b2"]
    output_df["green"] = df["sr_b3"]
    output_df["red"] = df["sr_b4"]
    # Vegetation indices
    output_df["vari_green_index"] = compute_vari_green_index(red=output_df["red"],green=output_df["green"],blue=output_df["blue"])
    # dates
    output_df["date (utc)"] = df["Landsat_overpass"].apply(lambda t: dt.datetime.combine(t.date(),dt.time(hour=18,minute=45,tzinfo=dt.UTC)))
    output_df["date"] = pd.to_datetime(df["Landsat_overpass"]) + pd.to_timedelta(output_df["local_time"], unit="s")
    # ET PT
    eastar = 6.13753*np.exp((17.27*output_df["ta"])/(output_df["ta"]+237.3))
    slope = 17.27 * 237.3 * eastar / (output_df["ta"] + 237.3) ** 2
    output_df["ep_pt"] = (PT_CST * slope * output_df["rn"]) / (slope + PSYCHROMETRIC_CST)

    # reorder columns
    return output_df[["name","date","date (utc)","lat","lon","ts","ta","td","rh","fc","lai","srn", "rn","ln","local_time","blue","green","red","nir","swir","vari_green_index","ep_pt"]]


In [ ]:
data = prepare_input_data(input_data)

In [ ]:
data.head()

In [ ]:
data["nir/swir"] = data["nir"]/data["swir"]
data["ts-ta"] = data["ts"] - data["ta"]

### Plot data

In [ ]:
plot_data(data,variable="fc")

## Use meteorological data and radiation data from EC tower

In [ ]:
data, ec_data = get_ec_data(data, stic_data_path)

In [ ]:
corrected_data = correct_data(data,ec_data,variables=["ta","td","rh","rn","srn"])

In [ ]:
corrected_data

## Compute initialization of STIC model

In [ ]:
def init_stic(df: pd.DataFrame, version:str)-> pd.DataFrame:
    """
    Compute STIC model 
    """
    res = run_batch_model(data=df,threshold=0.05,nb_steps=0,version=version)
    output_df = pd.DataFrame({
        "name": df["name"].values,
        "date": df["date"],
        "le": res[:,0],
        "h": res[:,1],
        "ef": res[:,2],
        "g": res[:,3],
        "rn": df["rn"].values,
        "ta": df["ta"].values,
        "td": df["td"].values,
        "rh": df["rh"].values,
        "ts": df["ts"].values,
        "ga": res[:,4],
        "gs": res[:,5],
        "t0": res[:,6],
        "m": res[:,7],
    })
    return output_df

In [ ]:
init_v1_3 = init_stic(corrected_data, version="1.3")
init_v1_3 = pd.merge(left=init_v1_3,right=ec_data,on=["date","name"])


In [ ]:
init_v1_3

In [ ]:
plot_metrics(init_v1_3,variable="le")

In [ ]:
init_v1_4 = init_stic(corrected_data,version="1.4")
init_v1_4 = pd.merge(left=init_v1_4,right=ec_data,on=["date","name"])

In [ ]:
plot_metrics(init_v1_4,variable="le")

In [ ]:
plot_scatter(init_v1_4,x="ec_le_closed",y="le")

## Compute STIC model

In [ ]:
def run_stic(df: pd.DataFrame, version:str)-> pd.DataFrame:
    """
    Compute STIC model 
    """
    res = run_batch_model(data=df,threshold=0.05,nb_steps=15,version=version)
    output_df = pd.DataFrame({
        "name": df["name"].values,
        "date": df["date"],
        "le": res[:,0],
        "h": res[:,1],
        "ef": res[:,2],
        "g": res[:,3],
        "rn": df["rn"].values,
        "ta": df["ta"].values,
        "td": df["td"].values,
        "rh": df["rh"].values,
        "ts": df["ts"].values,
        "ga": res[:,4],
        "gs": res[:,5],
        "t0": res[:,6],
        "m": res[:,7],
    })
    output_df["et"] = output_df.apply(lambda x: compute_et_from_le(x["le"]),axis=1)
    output_df["rn-g"] = output_df["rn"] - output_df["g"]
    return output_df

In [ ]:
estimated_v1_3 = run_stic(corrected_data,version="1.3")
res_v1_3 = pd.merge(left=estimated_v1_3,right=ec_data,on=["date","name"])

In [ ]:
estimated_v1_3

In [ ]:
estimated_v1_4 = run_stic(corrected_data, version="1.4")
res_v1_4 = pd.merge(left=estimated_v1_4,right=ec_data,on=["date","name"])

In [ ]:
estimated_v1_4

In [ ]:
plot_metrics(res_v1_3,variable="le")

In [ ]:
plot_metrics(res_v1_4, variable="le")

In [ ]:
plot_scatter(res_v1_4,x="ec_le_closed",y="le")

In [ ]:
corrected_data["green/red"] = corrected_data["green"]/corrected_data["red"]

In [ ]:
plot_scatter(corrected_data,x="ts-ta",y="green/red")

In [ ]:
res_v1_4["diff_le"] = res_v1_4["le"] - res_v1_4["ec_le"]

In [ ]:
plot_compare(corrected_data,res_v1_4, x="fc",y="diff_le")

In [ ]:
plot_compare(corrected_data,res_v1_4, x="ts-ta",y="diff_le")